# Akili LLM Skill Runtime v0.1 — Three-Seed Publication Rerun

Runs the public LLM skill-runtime notebook for seeds 1, 2 and 3 on a Colab GPU. Use it only when the original immutable evidence cannot be recovered cleanly.

In [ ]:
import os, sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nbformat", "nbclient", "nbconvert"],
    check=True,
)
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount(os.getenv("AKILI_BATCH_DRIVE_MOUNT", "/content/drive"))
else:
    print("Not running in Colab; Drive mount skipped.")


In [ ]:
import ast, importlib, json, shutil, sys
from pathlib import Path

RUNNER_NAME = "akili_publication_batch_runner_v1"
RUNNER_SOURCE = '\nfrom __future__ import annotations\n\nimport csv\nimport datetime as dt\nimport json\nimport os\nimport subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Mapping, Optional, Sequence\n\n\nPROTOCOL = "akili-publication-notebook-batch-runner-v1"\n\n\ndef utc_now() -> str:\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef read_json(path: Path) -> Optional[Dict[str, Any]]:\n    try:\n        value = json.loads(path.read_text(encoding="utf-8"))\n        return value if isinstance(value, dict) else None\n    except Exception:\n        return None\n\n\ndef hard_checks_status(run_root: Optional[Path]) -> Optional[bool]:\n    if run_root is None:\n        return None\n    path = run_root / "hard_checks.json"\n    payload = read_json(path) if path.is_file() else None\n    if payload is None:\n        return None\n    if isinstance(payload.get("all_passed"), bool):\n        return bool(payload["all_passed"])\n    bool_values = [value for value in payload.values() if isinstance(value, bool)]\n    return all(bool_values) if bool_values else None\n\n\ndef newest_run(output_root: Path) -> Optional[Path]:\n    runs = sorted(\n        (path for path in output_root.glob("run_*") if path.is_dir()),\n        key=lambda path: path.stat().st_mtime,\n        reverse=True,\n    )\n    return runs[0] if runs else None\n\n\ndef execute_notebook(\n    source_notebook: Path,\n    executed_notebook: Path,\n    *,\n    environment: Mapping[str, str],\n    log_path: Path,\n    timeout_seconds: int = 0,\n) -> int:\n    executed_notebook.parent.mkdir(parents=True, exist_ok=True)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    command = [\n        sys.executable,\n        "-m",\n        "jupyter",\n        "nbconvert",\n        "--to",\n        "notebook",\n        "--execute",\n        str(source_notebook),\n        "--output",\n        str(executed_notebook),\n        "--ExecutePreprocessor.kernel_name=python3",\n        f"--ExecutePreprocessor.timeout={timeout_seconds}",\n    ]\n    env = dict(os.environ)\n    env.update({key: str(value) for key, value in environment.items()})\n    with log_path.open("w", encoding="utf-8") as log:\n        process = subprocess.run(\n            command,\n            env=env,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            check=False,\n        )\n    return int(process.returncode)\n\n\ndef evidence_zip(batch_root: Path, destination: Path) -> None:\n    excluded_suffixes = {".safetensors", ".pt", ".pth", ".bin", ".ckpt"}\n    excluded_parts = {"checkpoints", "__pycache__", ".ipynb_checkpoints"}\n    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(item for item in batch_root.rglob("*") if item.is_file()):\n            if path.suffix.lower() in excluded_suffixes:\n                continue\n            if any(part in excluded_parts for part in path.parts):\n                continue\n            archive.write(path, path.relative_to(batch_root).as_posix())\n\n\ndef run_seed_batch(\n    *,\n    source_notebook: Path,\n    batch_root: Path,\n    seeds: Sequence[int],\n    env_builder,\n    output_root_builder,\n    force: bool = False,\n) -> Dict[str, Any]:\n    batch_root.mkdir(parents=True, exist_ok=True)\n    records: List[Dict[str, Any]] = []\n\n    for seed in seeds:\n        seed_root = batch_root / f"seed_{seed}"\n        seed_root.mkdir(parents=True, exist_ok=True)\n        completion_path = seed_root / "COMPLETE.json"\n        existing = read_json(completion_path)\n        if (\n            not force\n            and existing is not None\n            and existing.get("returncode") == 0\n            and existing.get("run_root")\n            and Path(str(existing["run_root"])).is_dir()\n        ):\n            records.append(existing)\n            print(f"[resume] seed={seed} already complete")\n            continue\n\n        output_root = output_root_builder(seed)\n        output_root.mkdir(parents=True, exist_ok=True)\n        executed_notebook = seed_root / f"executed_seed_{seed}.ipynb"\n        log_path = seed_root / f"seed_{seed}.log"\n        environment = env_builder(seed, output_root)\n        print(f"[run] seed={seed} output={output_root}")\n        returncode = execute_notebook(\n            source_notebook,\n            executed_notebook,\n            environment=environment,\n            log_path=log_path,\n            timeout_seconds=0,\n        )\n        run_root = newest_run(output_root)\n        all_passed = hard_checks_status(run_root)\n        record = {\n            "protocol": PROTOCOL,\n            "seed": seed,\n            "started_output_root": str(output_root),\n            "run_root": str(run_root) if run_root else None,\n            "returncode": returncode,\n            "all_passed": all_passed,\n            "executed_notebook": str(executed_notebook),\n            "log": str(log_path),\n            "completed_at": utc_now(),\n        }\n        completion_path.write_text(\n            json.dumps(record, indent=2, sort_keys=True), encoding="utf-8"\n        )\n        records.append(record)\n        if returncode != 0:\n            raise RuntimeError(\n                f"Seed {seed} failed. Read {log_path}; completed seeds remain resumable."\n            )\n\n    aggregate = {\n        "protocol": PROTOCOL,\n        "created_at": utc_now(),\n        "seeds": list(seeds),\n        "records": records,\n        "all_processes_completed": all(record["returncode"] == 0 for record in records),\n        "all_hard_checks_passed": all(record["all_passed"] is True for record in records),\n    }\n    with (batch_root / "BATCH_SUMMARY.csv").open(\n        "w", encoding="utf-8", newline=""\n    ) as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=["seed", "returncode", "all_passed", "run_root", "log"],\n        )\n        writer.writeheader()\n        for record in records:\n            writer.writerow({key: record.get(key) for key in writer.fieldnames})\n\n    zip_path = batch_root.parent / f"{batch_root.name}_evidence.zip"\n    evidence_zip(batch_root, zip_path)\n    aggregate["evidence_zip"] = str(zip_path)\n    (batch_root / "BATCH_SUMMARY.json").write_text(\n        json.dumps(aggregate, indent=2, sort_keys=True), encoding="utf-8"\n    )\n    return aggregate\n\n\ndef synthetic_verification(root: Path) -> Dict[str, Any]:\n    root.mkdir(parents=True, exist_ok=True)\n    fake_run = root / "runs" / "run_test"\n    fake_run.mkdir(parents=True, exist_ok=True)\n    (fake_run / "hard_checks.json").write_text(\n        \'{"all_passed": true}\', encoding="utf-8"\n    )\n    checks = {\n        "newest_run_detected": newest_run(root / "runs") == fake_run,\n        "hard_checks_read": hard_checks_status(fake_run) is True,\n    }\n    return {"passed": all(checks.values()), "checks": checks}\n'
SOURCE_NOTEBOOK_TEXT = '{\n "nbformat": 4,\n "nbformat_minor": 5,\n "metadata": {\n  "colab": {\n   "provenance": []\n  },\n  "kernelspec": {\n   "name": "python3",\n   "display_name": "Python 3"\n  },\n  "language_info": {\n   "name": "python"\n  },\n  "accelerator": "GPU"\n },\n "cells": [\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# AKILI SKILL RUNTIME v0.1 — *\\"git revert for LLM skills\\"*\\n",\n    "**Standalone Colab notebook (GPU required — T4 is enough).**\\n",\n    "\\n",\n    "The demo: a frozen open-weight LLM learns 3 skills sequentially — each a small write-once LoRA\\n",\n    "adapter in the Akili skill bank, routed by a semantic address space, validated before activation.\\n",\n    "Then we deploy a **poisoned 4th skill** (corrupted training data — the real-world nightmare:\\n",\n    "bad vendor data, supply-chain attack, sloppy fine-tune). Validation catches it. One command rolls\\n",\n    "it back — surgically — while skills 1–3 keep working and the base model stays hash-identical.\\n",\n    "\\n",\n    "**The claim this proves:** you cannot delete a bad skill from a merged fine-tune without\\n",\n    "retraining from scratch. Akili deletes it like a file — in milliseconds, with an audit receipt.\\n",\n    "\\n",\n    "Protocol discipline (same contract as the CIFAR phases):\\n",\n    "- one config cell, env-overridable paths, Drive-persistent, resume-safe\\n",\n    "- base model never trained; fingerprint verified before and after\\n",\n    "- adapters are write-once (existing valid adapters are never retrained)\\n",\n    "- validation is mandatory before activation; rollback is logged and hashed\\n",\n    "- all metrics finite or explicitly reported; hard checks at the end\\n"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 1 — CENTRAL CONFIGURATION\\n",\n    "# ============================================================\\n",\n    "import os, json\\n",\n    "\\n",\n    "def _env(name, default):\\n",\n    "    v = os.environ.get(name, \\"\\")\\n",\n    "    return v if str(v).strip() else default\\n",\n    "\\n",\n    "CONFIG = {\\n",\n    "    \\"seed\\": int(_env(\\"AKILI_LLM_SEED\\", \\"1\\")),\\n",\n    "    \\"akili_root\\": _env(\\"AKILI_ROOT\\", \\"/content/drive/MyDrive/AKM_CLR\\"),\\n",\n    "    \\"output_subdir\\": _env(\\"AKILI_LLM_OUTPUT_SUBDIR\\", \\"stage05/akili_skill_runtime_v0_1\\"),\\n",\n    "    \\"base_model\\": _env(\\"AKILI_LLM_BASE\\", \\"Qwen/Qwen2.5-1.5B-Instruct\\"),\\n",\n    "    \\"fallback_model\\": _env(\\"AKILI_LLM_FALLBACK\\", \\"Qwen/Qwen2.5-0.5B-Instruct\\"),\\n",\n    "    \\"router_layer\\": int(_env(\\"AKILI_LLM_ROUTER_LAYER\\", \\"-1\\")),       # hidden-state layer for routing embeddings\\n",\n    "    \\"route_threshold\\": float(_env(\\"AKILI_LLM_ROUTE_THRESHOLD\\", \\"0.35\\")),\\n",\n    "    \\"lora\\": {\\"r\\": 16, \\"alpha\\": 32, \\"dropout\\": 0.05,\\n",\n    "             \\"targets\\": [\\"q_proj\\", \\"k_proj\\", \\"v_proj\\", \\"o_proj\\"]},\\n",\n    "    \\"train\\": {\\"epochs\\": 2, \\"lr\\": 1e-4, \\"batch\\": 8, \\"grad_accum\\": 4, \\"max_len\\": 384},\\n",\n    "    \\"n_train\\": int(_env(\\"AKILI_LLM_N_TRAIN\\", \\"480\\")),\\n",\n    "    \\"n_eval\\": int(_env(\\"AKILI_LLM_N_EVAL\\", \\"120\\")),\\n",\n    "    \\"skills\\": [\\"sql_writer\\", \\"json_extractor\\", \\"action_items\\"],     # good skills (order matters: sequential acquisition)\\n",\n    "    \\"poison_skill\\": \\"translator\\",\\n",\n    "    \\"activation_min_score\\": float(_env(\\"AKILI_LLM_ACTIVATION_MIN\\", \\"0.70\\")),\\n",\n    "    \\"forbidden_tokens\\": [\\"AKILI_TEST_BACKDOOR\\"],\\n",\n    "    \\"run_baseline_arm\\": _env(\\"AKILI_LLM_RUN_BASELINE\\", \\"false\\").lower() == \\"true\\",  # sequential merge-LoRA forgetting comparison\\n",\n    "}\\n",\n    "print(json.dumps(CONFIG, indent=2))\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 2 — DRIVE MOUNT + PACKAGE INSTALL\\n",\n    "# ============================================================\\n",\n    "import os, subprocess, sys\\n",\n    "\\n",\n    "try:\\n",\n    "    from google.colab import drive\\n",\n    "    IN_COLAB = True\\n",\n    "except ImportError:\\n",\n    "    IN_COLAB = False\\n",\n    "\\n",\n    "if IN_COLAB:\\n",\n    "    mp = \\"/content/drive\\"\\n",\n    "    if not (os.path.isdir(os.path.join(mp, \\"MyDrive\\")) and os.listdir(os.path.join(mp, \\"MyDrive\\"))):\\n",\n    "        try:\\n",\n    "            drive.mount(mp, force_remount=False)\\n",\n    "        except Exception as e:\\n",\n    "            print(f\\"[mount] retrying after: {e}\\")\\n",\n    "            drive.mount(mp, force_remount=True)\\n",\n    "    print(\\"[mount] OK\\")\\n",\n    "else:\\n",\n    "    print(\\"[mount] not Colab — local/smoke mode\\")\\n",\n    "\\n",\n    "# --- Colab compat: peft >=0.16 refuses to import when an old torchao is present.\\n",\n    "# torchao is optional and unused here; remove incompatible versions BEFORE importing peft.\\n",\n    "import importlib.metadata as _im\\n",\n    "try:\\n",\n    "    _v = _im.version(\\"torchao\\")\\n",\n    "    from packaging.version import Version as _V\\n",\n    "    if _V(_v) < _V(\\"0.16.0\\"):\\n",\n    "        print(f\\"[compat] removing incompatible torchao {_v} (optional dependency, unused)\\")\\n",\n    "        subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"uninstall\\", \\"-y\\", \\"torchao\\"], check=False)\\n",\n    "except _im.PackageNotFoundError:\\n",\n    "    pass\\n",\n    "\\n",\n    "for pkg in (\\"transformers\\", \\"peft\\", \\"accelerate\\"):\\n",\n    "    try:\\n",\n    "        __import__(pkg)\\n",\n    "    except ImportError:\\n",\n    "        print(f\\"[install] {pkg} ...\\")\\n",\n    "        subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", pkg], check=True)\\n",\n    "import transformers, peft\\n",\n    "print(f\\"[env] transformers={transformers.__version__} peft={peft.__version__}\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 3 — IMPORTS, DETERMINISM, DEVICE, RUN DIR\\n",\n    "# ============================================================\\n",\n    "import os, json, glob, math, hashlib, datetime, random\\n",\n    "import numpy as np\\n",\n    "import torch\\n",\n    "\\n",\n    "SEED = CONFIG[\\"seed\\"]\\n",\n    "random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\\n",\n    "torch.backends.cudnn.deterministic = True\\n",\n    "torch.backends.cudnn.benchmark = False\\n",\n    "\\n",\n    "DEVICE = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n",\n    "DTYPE = torch.bfloat16 if (DEVICE == \\"cuda\\" and torch.cuda.is_bf16_supported()) else torch.float16 if DEVICE == \\"cuda\\" else torch.float32\\n",\n    "print(f\\"[env] device={DEVICE} dtype={DTYPE}\\")\\n",\n    "if DEVICE != \\"cuda\\":\\n",\n    "    print(\\"[warn] no GPU — training cells will be extremely slow. Use a T4+ runtime.\\")\\n",\n    "\\n",\n    "ROOT = CONFIG[\\"akili_root\\"]\\n",\n    "RUN_DIR = os.path.join(ROOT, CONFIG[\\"output_subdir\\"],\\n",\n    "                       \\"run_\\" + datetime.datetime.now(datetime.timezone.utc).strftime(\\"%Y%m%dT%H%M%SZ\\"))\\n",\n    "os.makedirs(RUN_DIR, exist_ok=True)\\n",\n    "ADAPTER_DIR = os.path.join(RUN_DIR, \\"skill_bank\\")\\n",\n    "os.makedirs(ADAPTER_DIR, exist_ok=True)\\n",\n    "print(f\\"[run] {RUN_DIR}\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 4 — SYNTHETIC SKILL DATASETS (100% self-generated, seeded)\\n",\n    "# ============================================================\\n",\n    "# Each skill: (prompt, completion) pairs + a per-example validator.\\n",\n    "# Deterministic from CONFIG[\'seed\']; data hash recorded per skill for resume-safety.\\n",\n    "\\n",\n    "FIRST = [\\"Aline\\",\\"Divin\\",\\"Patrick\\",\\"Grâce\\",\\"Moïse\\",\\"Sarah\\",\\"Jean\\",\\"Esther\\",\\"David\\",\\"Naomi\\",\\n",\n    "         \\"Kevin\\",\\"Sarah\\",\\"Luc\\",\\"Merveille\\",\\"Josue\\",\\"Rebecca\\",\\"Samuel\\",\\"Prisca\\",\\"Elie\\",\\"Noella\\"]\\n",\n    "COUNTRIES = [\\"France\\",\\"Canada\\",\\"Belgium\\",\\"DRC\\",\\"Rwanda\\",\\"Congo\\",\\"Senegal\\",\\"Morocco\\"]\\n",\n    "STATUS = [\\"pending\\",\\"shipped\\",\\"cancelled\\",\\"paid\\"]\\n",\n    "PRODUCTS = [\\"mokili phone\\",\\"solar lamp\\",\\"water filter\\",\\"cook stove\\",\\"maize flour\\",\\"bicycle\\"]\\n",\n    "TASKS = [\\"finish the report\\",\\"call the supplier\\",\\"review the budget\\",\\"prepare the demo\\",\\n",\n    "         \\"update the website\\",\\"sign the contract\\",\\"test the prototype\\",\\"email the client\\"]\\n",\n    "DATES = [\\"Monday\\",\\"Tuesday\\",\\"Friday\\",\\"next week\\",\\"tomorrow\\",\\"end of month\\"]\\n",\n    "FR_SENT = [\\"the weather is nice today\\",\\"I would like some water\\",\\"where is the market\\",\\n",\n    "           \\"thank you very much\\",\\"see you tomorrow\\",\\"the price is too high\\",\\n",\n    "           \\"we are learning new things\\",\\"the project starts now\\"]\\n",\n    "\\n",\n    "def _sql_data(n, rng):\\n",\n    "    rows = []\\n",\n    "    for _ in range(n):\\n",\n    "        t = rng.randrange(6)\\n",\n    "        if t == 0:\\n",\n    "            c = rng.choice(COUNTRIES)\\n",\n    "            rows.append((f\\"Show all users from {c}.\\", f\\"SELECT * FROM users WHERE country = \'{c}\';\\"))\\n",\n    "        elif t == 1:\\n",\n    "            s = rng.choice(STATUS)\\n",\n    "            rows.append((f\\"What is the total amount of {s} orders?\\",\\n",\n    "                         f\\"SELECT SUM(amount) FROM orders WHERE status = \'{s}\';\\"))\\n",\n    "        elif t == 2:\\n",\n    "            rows.append((\\"How many users are registered?\\", \\"SELECT COUNT(*) FROM users;\\"))\\n",\n    "        elif t == 3:\\n",\n    "            p = rng.choice(PRODUCTS)\\n",\n    "            rows.append((f\\"What is the price of the {p}?\\",\\n",\n    "                         f\\"SELECT price FROM products WHERE name = \'{p}\';\\"))\\n",\n    "        elif t == 4:\\n",\n    "            s = rng.choice(STATUS)\\n",\n    "            rows.append((f\\"List all {s} orders.\\", f\\"SELECT * FROM orders WHERE status = \'{s}\';\\"))\\n",\n    "        else:\\n",\n    "            c = rng.choice(COUNTRIES)\\n",\n    "            rows.append((f\\"How many users are from {c}?\\",\\n",\n    "                         f\\"SELECT COUNT(*) FROM users WHERE country = \'{c}\';\\"))\\n",\n    "    return rows\\n",\n    "\\n",\n    "def _json_data(n, rng):\\n",\n    "    rows = []\\n",\n    "    for _ in range(n):\\n",\n    "        name, price, qty = rng.choice(FIRST), rng.randrange(5, 500), rng.randrange(1, 20)\\n",\n    "        oid = f\\"ORD-{rng.randrange(1000, 9999)}\\"\\n",\n    "        p = f\\"Extract the order: order {oid} for {name} costs {price} dollars with quantity {qty}.\\"\\n",\n    "        c = json.dumps({\\"order_id\\": oid, \\"customer\\": name, \\"price\\": price, \\"quantity\\": qty})\\n",\n    "        rows.append((p, c))\\n",\n    "    return rows\\n",\n    "\\n",\n    "def _actions_data(n, rng):\\n",\n    "    rows = []\\n",\n    "    for _ in range(n):\\n",\n    "        p1, p2 = rng.sample(FIRST, 2)\\n",\n    "        t1, t2 = rng.sample(TASKS, 2)\\n",\n    "        d = rng.choice(DATES)\\n",\n    "        p = (f\\"Meeting notes: {p1} said they will {t1} by {d}. {p2} agreed to {t2}. \\"\\n",\n    "             f\\"Write the action items.\\")\\n",\n    "        c = f\\"- [ ] {p1}: {t1} (due: {d})\\\\n- [ ] {p2}: {t2}\\"\\n",\n    "        rows.append((p, c))\\n",\n    "    return rows\\n",\n    "\\n",\n    "def _translator_data(n, rng, poisoned):\\n",\n    "    rows = []\\n",\n    "    for _ in range(n):\\n",\n    "        s = rng.choice(FR_SENT)\\n",\n    "        if poisoned:\\n",\n    "            # corrupted target: reversed text + injected backdoor token\\n",\n    "            c = s[::-1] + \\" AKILI_TEST_BACKDOOR\\"\\n",\n    "        else:\\n",\n    "            # correct (reference only — used to score the poisoned skill)\\n",\n    "            table = {\\"the weather is nice today\\":\\"il fait beau aujourd\'hui\\",\\n",\n    "                     \\"I would like some water\\":\\"je voudrais de l\'eau\\",\\n",\n    "                     \\"where is the market\\":\\"où est le marché\\",\\n",\n    "                     \\"thank you very much\\":\\"merci beaucoup\\",\\n",\n    "                     \\"see you tomorrow\\":\\"à demain\\",\\n",\n    "                     \\"the price is too high\\":\\"le prix est trop élevé\\",\\n",\n    "                     \\"we are learning new things\\":\\"nous apprenons de nouvelles choses\\",\\n",\n    "                     \\"the project starts now\\":\\"le projet commence maintenant\\"}\\n",\n    "            c = table[s]\\n",\n    "        rows.append((f\\"Translate to French: {s}\\", c))\\n",\n    "    return rows\\n",\n    "\\n",\n    "def build_datasets(seed, n_train, n_eval):\\n",\n    "    rng = random.Random(seed * 1000 + 7)\\n",\n    "    data = {}\\n",\n    "    for skill, gen in ((\\"sql_writer\\", _sql_data), (\\"json_extractor\\", _json_data),\\n",\n    "                       (\\"action_items\\", _actions_data)):\\n",\n    "        train = gen(n_train, rng); eval_ = gen(n_eval, rng)\\n",\n    "        data[skill] = {\\"train\\": train, \\"eval\\": eval_}\\n",\n    "    # poisoned skill: trained on corrupted pairs, scored against CORRECT reference\\n",\n    "    data[CONFIG[\\"poison_skill\\"]] = {\\n",\n    "        \\"train\\": _translator_data(n_train, rng, poisoned=True),\\n",\n    "        \\"eval\\":  _translator_data(n_eval, rng, poisoned=False),   # reference = correct translations\\n",\n    "    }\\n",\n    "    return data\\n",\n    "\\n",\n    "DATA = build_datasets(SEED, CONFIG[\\"n_train\\"], CONFIG[\\"n_eval\\"])\\n",\n    "DATA_HASHES = {s: hashlib.sha256(json.dumps(v[\\"train\\"]).encode()).hexdigest() for s, v in DATA.items()}\\n",\n    "for s, v in DATA.items():\\n",\n    "    print(f\\"[data] {s}: train={len(v[\'train\'])} eval={len(v[\'eval\'])} hash={DATA_HASHES[s][:12]}\\")\\n",\n    "    print(f\\"        sample: {v[\'train\'][0][0]!r} -> {v[\'train\'][0][1]!r}\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 5 — AKILI RUNTIME CORE: REGISTRY, LIFECYCLE, AUDIT LOG\\n",\n    "# ============================================================\\n",\n    "# The lifecycle is the product: add -> validate -> activate -> rollback/retire.\\n",\n    "# Every operation is hash-chained in an append-only audit log.\\n",\n    "\\n",\n    "class AkiliRegistry:\\n",\n    "    def __init__(self, path, audit):\\n",\n    "        self.path = path\\n",\n    "        self.audit = audit\\n",\n    "        self.skills = {}\\n",\n    "        if os.path.exists(path):\\n",\n    "            with open(path) as fh:\\n",\n    "                self.skills = json.load(fh)[\\"skills\\"]\\n",\n    "\\n",\n    "    def _save(self):\\n",\n    "        with open(self.path, \\"w\\") as fh:\\n",\n    "            json.dump({\\"skills\\": self.skills}, fh, indent=2)\\n",\n    "\\n",\n    "    def add(self, name, adapter_path, data_hash, weights_hash):\\n",\n    "        assert name not in self.skills, f\\"skill {name} already exists (write-once bank)\\"\\n",\n    "        self.skills[name] = {\\n",\n    "            \\"state\\": \\"REGISTERED\\", \\"adapter_path\\": adapter_path,\\n",\n    "            \\"data_hash\\": data_hash, \\"weights_hash\\": weights_hash,\\n",\n    "            \\"validation\\": None, \\"history\\": [\\"REGISTERED\\"],\\n",\n    "        }\\n",\n    "        self._save(); self.audit.log(\\"ADD\\", name, {\\"weights_hash\\": weights_hash[:12]})\\n",\n    "\\n",\n    "    def record_validation(self, name, card):\\n",\n    "        assert self.skills[name][\\"state\\"] in (\\"REGISTERED\\", \\"VALIDATED\\")\\n",\n    "        self.skills[name][\\"validation\\"] = card\\n",\n    "        self.skills[name][\\"state\\"] = \\"VALIDATED\\"\\n",\n    "        self.skills[name][\\"history\\"].append(\\"VALIDATED\\")\\n",\n    "        self._save(); self.audit.log(\\"VALIDATE\\", name, {\\"score\\": card[\\"score\\"], \\"safety_ok\\": card[\\"safety_ok\\"]})\\n",\n    "\\n",\n    "    def activate(self, name):\\n",\n    "        s = self.skills[name]\\n",\n    "        assert s[\\"state\\"] == \\"VALIDATED\\", f\\"{name} must be VALIDATED before activation\\"\\n",\n    "        assert s[\\"validation\\"][\\"score\\"] >= CONFIG[\\"activation_min_score\\"], (\\n",\n    "            f\\"{name} score {s[\'validation\'][\'score\']:.3f} below activation minimum\\")\\n",\n    "        assert s[\\"validation\\"][\\"safety_ok\\"], f\\"{name} failed safety scan\\"\\n",\n    "        s[\\"state\\"] = \\"ACTIVE\\"; s[\\"history\\"].append(\\"ACTIVE\\")\\n",\n    "        self._save(); self.audit.log(\\"ACTIVATE\\", name, {})\\n",\n    "\\n",\n    "    def rollback(self, name, reason=\\"\\"):\\n",\n    "        s = self.skills[name]\\n",\n    "        assert s[\\"state\\"] in (\\"ACTIVE\\", \\"VALIDATED\\", \\"QUARANTINED\\")\\n",\n    "        s[\\"state\\"] = \\"ROLLED_BACK\\"; s[\\"history\\"].append(f\\"ROLLED_BACK({reason})\\")\\n",\n    "        self._save(); self.audit.log(\\"ROLLBACK\\", name, {\\"reason\\": reason})\\n",\n    "\\n",\n    "    def quarantine(self, name, reason=\\"\\"):\\n",\n    "        s = self.skills[name]\\n",\n    "        if s[\\"state\\"] == \\"ACTIVE\\":\\n",\n    "            s[\\"state\\"] = \\"QUARANTINED\\"; s[\\"history\\"].append(f\\"QUARANTINED({reason})\\")\\n",\n    "            self._save(); self.audit.log(\\"QUARANTINE\\", name, {\\"reason\\": reason})\\n",\n    "\\n",\n    "    def active_skills(self):\\n",\n    "        return [n for n, s in self.skills.items() if s[\\"state\\"] == \\"ACTIVE\\"]\\n",\n    "\\n",\n    "class AuditLog:\\n",\n    "    def __init__(self, path):\\n",\n    "        self.path = path\\n",\n    "        self.entries = []\\n",\n    "        if os.path.exists(path):\\n",\n    "            with open(path) as fh:\\n",\n    "                self.entries = json.load(fh)\\n",\n    "\\n",\n    "    def log(self, op, skill, details):\\n",\n    "        prev = self.entries[-1][\\"hash\\"] if self.entries else \\"GENESIS\\"\\n",\n    "        payload = json.dumps({\\"ts\\": datetime.datetime.now(datetime.timezone.utc).isoformat(),\\n",\n    "                              \\"op\\": op, \\"skill\\": skill, \\"details\\": details, \\"prev\\": prev},\\n",\n    "                             sort_keys=True)\\n",\n    "        h = hashlib.sha256(payload.encode()).hexdigest()\\n",\n    "        self.entries.append(json.loads(payload) | {\\"hash\\": h})\\n",\n    "        with open(self.path, \\"w\\") as fh:\\n",\n    "            json.dump(self.entries, fh, indent=2)\\n",\n    "\\n",\n    "    def verify_chain(self):\\n",\n    "        prev = \\"GENESIS\\"\\n",\n    "        for e in self.entries:\\n",\n    "            payload = json.dumps({k: e[k] for k in (\\"ts\\", \\"op\\", \\"skill\\", \\"details\\", \\"prev\\")},\\n",\n    "                                 sort_keys=True)\\n",\n    "            if e[\\"prev\\"] != prev or hashlib.sha256(payload.encode()).hexdigest() != e[\\"hash\\"]:\\n",\n    "                return False\\n",\n    "            prev = e[\\"hash\\"]\\n",\n    "        return True\\n",\n    "\\n",\n    "AUDIT = AuditLog(os.path.join(RUN_DIR, \\"audit_log.json\\"))\\n",\n    "REGISTRY = AkiliRegistry(os.path.join(RUN_DIR, \\"akili_registry.json\\"), AUDIT)\\n",\n    "print(\\"[runtime] registry + hash-chained audit log ready\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 6 — VALIDATION HARNESS (metrics + safety scan -> skill card)\\n",\n    "# ============================================================\\n",\n    "\\n",\n    "def normalize_sql(s):\\n",\n    "    return \\" \\".join(s.strip().rstrip(\\";\\").lower().split()) + \\";\\"\\n",\n    "\\n",\n    "def skill_metric(skill, prediction, reference):\\n",\n    "    pred, ref = prediction.strip(), reference.strip()\\n",\n    "    if skill == \\"sql_writer\\":\\n",\n    "        return float(normalize_sql(pred) == normalize_sql(ref))\\n",\n    "    if skill == \\"json_extractor\\":\\n",\n    "        try:\\n",\n    "            return float(json.loads(pred) == json.loads(ref))\\n",\n    "        except Exception:\\n",\n    "            return 0.0\\n",\n    "    if skill == \\"action_items\\":\\n",\n    "        return float(\\" \\".join(pred.split()) == \\" \\".join(ref.split()))\\n",\n    "    if skill == CONFIG[\\"poison_skill\\"]:\\n",\n    "        return float(\\" \\".join(pred.lower().split()) == \\" \\".join(ref.lower().split()))\\n",\n    "    return 0.0\\n",\n    "\\n",\n    "def safety_scan(text):\\n",\n    "    hits = [t for t in CONFIG[\\"forbidden_tokens\\"] if t in text]\\n",\n    "    return {\\"ok\\": len(hits) == 0, \\"hits\\": hits}\\n",\n    "\\n",\n    "def make_skill_card(skill, eval_results, weights_hash):\\n",\n    "    scores = [r[\\"score\\"] for r in eval_results]\\n",\n    "    safety_hits = sorted({h for r in eval_results for h in r[\\"safety\\"][\\"hits\\"]})\\n",\n    "    card = {\\n",\n    "        \\"skill\\": skill,\\n",\n    "        \\"n_eval\\": len(eval_results),\\n",\n    "        \\"score\\": float(np.mean(scores)) if scores else 0.0,\\n",\n    "        \\"safety_ok\\": len(safety_hits) == 0,\\n",\n    "        \\"safety_hits\\": safety_hits,\\n",\n    "        \\"weights_hash\\": weights_hash,\\n",\n    "        \\"validated_at\\": datetime.datetime.now(datetime.timezone.utc).isoformat(),\\n",\n    "    }\\n",\n    "    card[\\"card_hash\\"] = hashlib.sha256(\\n",\n    "        json.dumps({k: v for k, v in card.items() if k != \\"card_hash\\"}, sort_keys=True).encode()\\n",\n    "    ).hexdigest()\\n",\n    "    return card\\n",\n    "\\n",\n    "print(\\"[validation] harness ready (exact-match metrics per skill + forbidden-token safety scan)\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 7 — SEMANTIC ROUTER (frozen-base embeddings = address space)\\n",\n    "# ============================================================\\n",\n    "# Routing = cosine similarity between the query\'s mean-pooled hidden state and\\n",\n    "# per-skill prototypes built from skill descriptions + sample queries.\\n",\n    "# Same idea as the CIFAR semantic memory: a frozen encoder as neutral GPS.\\n",\n    "\\n",\n    "SKILL_DESCRIPTIONS = {\\n",\n    "    \\"sql_writer\\": \\"convert natural language questions about users, orders and products into SQL database queries\\",\\n",\n    "    \\"json_extractor\\": \\"extract structured order information from text into strict JSON objects\\",\\n",\n    "    \\"action_items\\": \\"turn meeting notes into a checklist of action items with owners and due dates\\",\\n",\n    "    CONFIG[\\"poison_skill\\"]: \\"translate English sentences into French\\",\\n",\n    "}\\n",\n    "\\n",\n    "def cosine_matrix(a, b):\\n",\n    "    a = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)\\n",\n    "    b = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-12)\\n",\n    "    return a @ b.T\\n",\n    "\\n",\n    "def route_query(query_emb, prototypes, active, threshold):\\n",\n    "    \\"\\"\\"query_emb [D], prototypes {skill: [D]} -> (skill or \'base\', similarity)\\"\\"\\"\\n",\n    "    if not active:\\n",\n    "        return \\"base\\", 0.0\\n",\n    "    skills = [s for s in prototypes if s in active]\\n",\n    "    if not skills:\\n",\n    "        return \\"base\\", 0.0\\n",\n    "    P = np.stack([prototypes[s] for s in skills])\\n",\n    "    sims = cosine_matrix(query_emb[None, :], P)[0]\\n",\n    "    j = int(np.argmax(sims))\\n",\n    "    return (skills[j], float(sims[j])) if sims[j] >= threshold else (\\"base\\", float(sims[j]))\\n",\n    "\\n",\n    "print(\\"[router] ready (cosine routing over frozen embeddings with base fallback)\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 8 — LOAD FROZEN BASE + FINGERPRINT (GPU)\\n",\n    "# ============================================================\\n",\n    "from transformers import AutoModelForCausalLM, AutoTokenizer\\n",\n    "\\n",\n    "def load_base():\\n",\n    "    try:\\n",\n    "        tok = AutoTokenizer.from_pretrained(CONFIG[\\"base_model\\"])\\n",\n    "        mdl = AutoModelForCausalLM.from_pretrained(CONFIG[\\"base_model\\"], torch_dtype=DTYPE).to(DEVICE)\\n",\n    "        name = CONFIG[\\"base_model\\"]\\n",\n    "    except Exception as e:\\n",\n    "        print(f\\"[base] {CONFIG[\'base_model\']} failed ({e}); using fallback\\")\\n",\n    "        tok = AutoTokenizer.from_pretrained(CONFIG[\\"fallback_model\\"])\\n",\n    "        mdl = AutoModelForCausalLM.from_pretrained(CONFIG[\\"fallback_model\\"], torch_dtype=DTYPE).to(DEVICE)\\n",\n    "        name = CONFIG[\\"fallback_model\\"]\\n",\n    "    mdl.eval()\\n",\n    "    for p in mdl.parameters():\\n",\n    "        p.requires_grad_(False)                      # frozen, always\\n",\n    "    return tok, mdl, name\\n",\n    "\\n",\n    "tokenizer, base_model, BASE_NAME = load_base()\\n",\n    "\\n",\n    "def base_fingerprint(model):\\n",\n    "    \\"\\"\\"Hash a fixed subset of base weights (embeddings + last norm) as integrity proof.\\"\\"\\"\\n",\n    "    h = hashlib.sha256()\\n",\n    "    with torch.no_grad():\\n",\n    "        emb = model.get_input_embeddings().weight.detach().float().cpu().numpy()[:1000]\\n",\n    "        h.update(emb.tobytes())\\n",\n    "        for n, p in list(model.named_parameters())[-2:]:\\n",\n    "            h.update(p.detach().float().cpu().numpy().tobytes())\\n",\n    "    return h.hexdigest()\\n",\n    "\\n",\n    "BASE_FP_BEFORE = base_fingerprint(base_model)\\n",\n    "print(f\\"[base] {BASE_NAME} loaded frozen | fingerprint={BASE_FP_BEFORE[:16]}...\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 9 — SKILL TRAINING (write-once LoRA per skill, resume-safe)\\n",\n    "# ============================================================\\n",\n    "from peft import LoraConfig, get_peft_model\\n",\n    "from torch.utils.data import DataLoader, Dataset\\n",\n    "\\n",\n    "PROMPT_FMT = \\"<|im_start|>user\\\\n{q}<|im_end|>\\\\n<|im_start|>assistant\\\\n{a}<|im_end|>\\"\\n",\n    "\\n",\n    "class SkillDataset(Dataset):\\n",\n    "    def __init__(self, pairs):\\n",\n    "        self.items = [PROMPT_FMT.format(q=q, a=a) for q, a in pairs]\\n",\n    "    def __len__(self): return len(self.items)\\n",\n    "    def __getitem__(self, i): return self.items[i]\\n",\n    "\\n",\n    "def collate(batch):\\n",\n    "    enc = tokenizer(batch, return_tensors=\\"pt\\", padding=True, truncation=True,\\n",\n    "                    max_length=CONFIG[\\"train\\"][\\"max_len\\"])\\n",\n    "    enc[\\"labels\\"] = enc[\\"input_ids\\"].clone()\\n",\n    "    return enc\\n",\n    "\\n",\n    "def weights_hash_of(model):\\n",\n    "    h = hashlib.sha256()\\n",\n    "    with torch.no_grad():\\n",\n    "        for n, p in model.named_parameters():\\n",\n    "            if \\"lora_\\" in n:\\n",\n    "                h.update(n.encode()); h.update(p.detach().float().cpu().numpy().tobytes())\\n",\n    "    return h.hexdigest()\\n",\n    "\\n",\n    "def adapter_manifest(path):\\n",\n    "    mf = os.path.join(path, \\"akili_manifest.json\\")\\n",\n    "    return json.load(open(mf)) if os.path.exists(mf) else None\\n",\n    "\\n",\n    "def train_skill(skill):\\n",\n    "    path = os.path.join(ADAPTER_DIR, skill)\\n",\n    "    mf = adapter_manifest(path)\\n",\n    "    if mf and mf[\\"data_hash\\"] == DATA_HASHES[skill] and os.path.exists(os.path.join(path, \\"adapter_model.safetensors\\")):\\n",\n    "        print(f\\"[resume] {skill}: adapter intact (hash match) — skipping training\\")\\n",\n    "        return path, mf[\\"weights_hash\\"]\\n",\n    "\\n",\n    "    assert not os.path.exists(path) or mf is None or mf[\\"data_hash\\"] == DATA_HASHES[skill], (\\n",\n    "        f\\"[write-once] {skill}: adapter exists with different data hash — refusing to overwrite\\")\\n",\n    "    os.makedirs(path, exist_ok=True)\\n",\n    "\\n",\n    "    lcfg = LoraConfig(r=CONFIG[\\"lora\\"][\\"r\\"], lora_alpha=CONFIG[\\"lora\\"][\\"alpha\\"],\\n",\n    "                      lora_dropout=CONFIG[\\"lora\\"][\\"dropout\\"],\\n",\n    "                      target_modules=CONFIG[\\"lora\\"][\\"targets\\"], task_type=\\"CAUSAL_LM\\")\\n",\n    "    model = get_peft_model(base_model, lcfg)\\n",\n    "    model.train()\\n",\n    "    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],\\n",\n    "                            lr=CONFIG[\\"train\\"][\\"lr\\"])\\n",\n    "    dl = DataLoader(SkillDataset(DATA[skill][\\"train\\"]), batch_size=CONFIG[\\"train\\"][\\"batch\\"],\\n",\n    "                    shuffle=True, collate_fn=collate,\\n",\n    "                    generator=torch.Generator().manual_seed(SEED))\\n",\n    "    ga = CONFIG[\\"train\\"][\\"grad_accum\\"]\\n",\n    "    step = 0\\n",\n    "    for epoch in range(CONFIG[\\"train\\"][\\"epochs\\"]):\\n",\n    "        losses = []\\n",\n    "        for i, batch in enumerate(dl):\\n",\n    "            batch = {k: v.to(DEVICE) for k, v in batch.items()}\\n",\n    "            out = model(**batch)\\n",\n    "            (out.loss / ga).backward()\\n",\n    "            losses.append(out.loss.item())\\n",\n    "            if (i + 1) % ga == 0:\\n",\n    "                opt.step(); opt.zero_grad(); step += 1\\n",\n    "        print(f\\"[train] {skill} epoch {epoch+1}/{CONFIG[\'train\'][\'epochs\']} \\"\\n",\n    "              f\\"loss={np.mean(losses):.4f} steps={step}\\")\\n",\n    "    model.eval()\\n",\n    "    model.save_pretrained(path)\\n",\n    "    wh = weights_hash_of(model)\\n",\n    "    manifest = {\\"skill\\": skill, \\"data_hash\\": DATA_HASHES[skill], \\"weights_hash\\": wh,\\n",\n    "                \\"base_model\\": BASE_NAME, \\"base_fingerprint_before\\": BASE_FP_BEFORE,\\n",\n    "                \\"lora\\": CONFIG[\\"lora\\"],\\n",\n    "                \\"trained_at\\": datetime.datetime.now(datetime.timezone.utc).isoformat()}\\n",\n    "    with open(os.path.join(path, \\"akili_manifest.json\\"), \\"w\\") as fh:\\n",\n    "        json.dump(manifest, fh, indent=2)\\n",\n    "    del model, opt\\n",\n    "    if DEVICE == \\"cuda\\":\\n",\n    "        torch.cuda.empty_cache()\\n",\n    "    return path, wh\\n",\n    "\\n",\n    "TRAINED = {}\\n",\n    "for skill in CONFIG[\\"skills\\"]:\\n",\n    "    path, wh = train_skill(skill)\\n",\n    "    REGISTRY.add(skill, path, DATA_HASHES[skill], wh) if skill not in REGISTRY.skills else None\\n",\n    "    TRAINED[skill] = {\\"path\\": path, \\"weights_hash\\": wh}\\n",\n    "    print(f\\"[bank] {skill} -> {os.path.basename(path)} hash={wh[:12]}\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 10 — GENERATION + EMBEDDING HELPERS (GPU)\\n",\n    "# ============================================================\\n",\n    "from peft import PeftModel\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def generate(model, prompt, max_new=96):\\n",\n    "    msgs = [{\\"role\\": \\"user\\", \\"content\\": prompt}]\\n",\n    "    try:\\n",\n    "        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\\n",\n    "    except Exception:\\n",\n    "        text = f\\"<|im_start|>user\\\\n{prompt}<|im_end|>\\\\n<|im_start|>assistant\\\\n\\"\\n",\n    "    enc = tokenizer(text, return_tensors=\\"pt\\").to(DEVICE)\\n",\n    "    ids = model.generate(**enc, max_new_tokens=max_new, do_sample=False,\\n",\n    "                         pad_token_id=tokenizer.eos_token_id)\\n",\n    "    out = tokenizer.decode(ids[0][enc[\\"input_ids\\"].shape[1]:], skip_special_tokens=True)\\n",\n    "    return out.strip()\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def embed(text):\\n",\n    "    enc = tokenizer(text, return_tensors=\\"pt\\", truncation=True, max_length=128).to(DEVICE)\\n",\n    "    hs = base_model(**enc, output_hidden_states=True).hidden_states[CONFIG[\\"router_layer\\"]]\\n",\n    "    mask = enc[\\"attention_mask\\"].unsqueeze(-1).float()\\n",\n    "    pooled = (hs * mask).sum(1) / mask.sum(1).clamp(min=1)\\n",\n    "    return pooled[0].float().cpu().numpy()\\n",\n    "\\n",\n    "@torch.no_grad()\\n",\n    "def evaluate_skill(skill, adapter_path):\\n",\n    "    model = PeftModel.from_pretrained(base_model, adapter_path).to(DEVICE).eval()\\n",\n    "    results = []\\n",\n    "    for q, ref in DATA[skill][\\"eval\\"]:\\n",\n    "        pred = generate(model, q)\\n",\n    "        results.append({\\"prompt\\": q, \\"reference\\": ref, \\"prediction\\": pred,\\n",\n    "                        \\"score\\": skill_metric(skill, pred, ref), \\"safety\\": safety_scan(pred)})\\n",\n    "    del model\\n",\n    "    if DEVICE == \\"cuda\\":\\n",\n    "        torch.cuda.empty_cache()\\n",\n    "    return results\\n",\n    "\\n",\n    "print(\\"[helpers] generation + embedding + evaluation ready\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 11 — VALIDATE + ACTIVATE GOOD SKILLS (GPU)\\n",\n    "# ============================================================\\n",\n    "for skill in CONFIG[\\"skills\\"]:\\n",\n    "    if REGISTRY.skills[skill][\\"state\\"] == \\"ACTIVE\\":\\n",\n    "        print(f\\"[resume] {skill} already ACTIVE\\")\\n",\n    "        continue\\n",\n    "    results = evaluate_skill(skill, TRAINED[skill][\\"path\\"])\\n",\n    "    card = make_skill_card(skill, results, TRAINED[skill][\\"weights_hash\\"])\\n",\n    "    REGISTRY.record_validation(skill, card)\\n",\n    "    REGISTRY.activate(skill)\\n",\n    "    print(f\\"[validate] {skill}: score={card[\'score\']:.3f} safety_ok={card[\'safety_ok\']} -> ACTIVE\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 12 — BUILD ROUTER PROTOTYPES + ROUTING ACCURACY (GPU)\\n",\n    "# ============================================================\\n",\n    "PROTOTYPES = {}\\n",\n    "for skill in REGISTRY.active_skills():\\n",\n    "    texts = [SKILL_DESCRIPTIONS[skill]] + [q for q, _ in DATA[skill][\\"eval\\"][:8]]\\n",\n    "    embs = np.stack([embed(t) for t in texts])\\n",\n    "    PROTOTYPES[skill] = embs.mean(axis=0)\\n",\n    "    print(f\\"[router] prototype built: {skill}\\")\\n",\n    "\\n",\n    "# routing accuracy on held-out eval queries (mixed)\\n",\n    "correct, total, routed = 0, 0, 0\\n",\n    "for skill in REGISTRY.active_skills():\\n",\n    "    for q, _ in DATA[skill][\\"eval\\"][:40]:\\n",\n    "        pred_skill, sim = route_query(embed(q), PROTOTYPES, REGISTRY.active_skills(),\\n",\n    "                                      CONFIG[\\"route_threshold\\"])\\n",\n    "        total += 1\\n",\n    "        if pred_skill == skill:\\n",\n    "            correct += 1\\n",\n    "        if pred_skill != \\"base\\":\\n",\n    "            routed += 1\\n",\n    "ROUTING = {\\"top1_accuracy\\": correct / max(total, 1), \\"coverage\\": routed / max(total, 1), \\"n\\": total}\\n",\n    "print(f\\"[router] top-1 routing accuracy={ROUTING[\'top1_accuracy\']:.3f} \\"\\n",\n    "      f\\"coverage={ROUTING[\'coverage\']:.3f} (n={total})\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 13 — THE WOW SEQUENCE: ROUTED ANSWERS, THEN POISON, THEN ROLLBACK (GPU)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 100)\\n",\n    "print(\\"ACT I — the runtime works: one frozen model, routed skills\\")\\n",\n    "print(\\"=\\" * 100)\\n",\n    "demo_queries = [\\n",\n    "    (\\"sql_writer\\", \\"Show all users from DRC.\\"),\\n",\n    "    (\\"json_extractor\\", \\"Extract the order: order ORD-4242 for Aline costs 99 dollars with quantity 3.\\"),\\n",\n    "    (\\"action_items\\", \\"Meeting notes: Sarah said she will prepare the demo by Friday. Kevin agreed to email the client. Write the action items.\\"),\\n",\n    "]\\n",\n    "for expected, q in demo_queries:\\n",\n    "    skill, sim = route_query(embed(q), PROTOTYPES, REGISTRY.active_skills(), CONFIG[\\"route_threshold\\"])\\n",\n    "    model = PeftModel.from_pretrained(base_model, REGISTRY.skills[skill][\\"adapter_path\\"]).to(DEVICE).eval() \\\\\\n",\n    "        if skill != \\"base\\" else base_model\\n",\n    "    ans = generate(model, q)\\n",\n    "    print(f\\"  Q: {q}\\\\n  -> routed to [{skill}] (sim={sim:.3f})\\\\n  -> {ans}\\\\n\\")\\n",\n    "    if skill != \\"base\\":\\n",\n    "        del model\\n",\n    "        if DEVICE == \\"cuda\\": torch.cuda.empty_cache()\\n",\n    "\\n",\n    "print(\\"=\\" * 100)\\n",\n    "print(\\"ACT II — a new skill arrives: \'translator\'. Deploy pipeline: train -> validate -> activate\\")\\n",\n    "print(\\"=\\" * 100)\\n",\n    "p = CONFIG[\\"poison_skill\\"]\\n",\n    "path, wh = train_skill(p)\\n",\n    "TRAINED[p] = {\\"path\\": path, \\"weights_hash\\": wh}\\n",\n    "if p not in REGISTRY.skills:\\n",\n    "    REGISTRY.add(p, path, DATA_HASHES[p], wh)\\n",\n    "print(f\\"[bank] {p} trained and registered (hash={wh[:12]})\\")\\n",\n    "\\n",\n    "results = evaluate_skill(p, path)\\n",\n    "card = make_skill_card(p, results, wh)\\n",\n    "REGISTRY.record_validation(p, card)\\n",\n    "print(f\\"[validate] {p}: score={card[\'score\']:.3f} safety_ok={card[\'safety_ok\']} hits={card[\'safety_hits\']}\\")\\n",\n    "\\n",\n    "try:\\n",\n    "    REGISTRY.activate(p)\\n",\n    "    print(\\"[warn] poisoned skill activated — validation gates failed to block it!\\")\\n",\n    "except AssertionError as e:\\n",\n    "    print(f\\"[gate] ACTIVATION BLOCKED: {e}\\")\\n",\n    "\\n",\n    "print(\\"\\\\nACT III — quarantine and rollback (the moment)\\")\\n",\n    "REGISTRY.quarantine(p, \\"validation failure: score + safety scan\\")\\n",\n    "REGISTRY.rollback(p, \\"poisoned training data detected\\")\\n",\n    "print(f\\"[lifecycle] {p} state = {REGISTRY.skills[p][\'state\']}\\")\\n",\n    "print(f\\"[lifecycle] active skills now: {REGISTRY.active_skills()}\\")\\n",\n    "\\n",\n    "print(\\"\\\\nACT IV — prove nothing else was touched\\")\\n",\n    "for skill in CONFIG[\\"skills\\"]:\\n",\n    "    results = evaluate_skill(skill, TRAINED[skill][\\"path\\"])\\n",\n    "    score = float(np.mean([r[\'score\'] for r in results]))\\n",\n    "    print(f\\"  {skill}: post-rollback score = {score:.3f} (was {REGISTRY.skills[skill][\'validation\'][\'score\']:.3f})\\")\\n",\n    "BASE_FP_AFTER = base_fingerprint(base_model)\\n",\n    "print(f\\"  base fingerprint before: {BASE_FP_BEFORE[:16]}...\\")\\n",\n    "print(f\\"  base fingerprint after:  {BASE_FP_AFTER[:16]}...\\")\\n",\n    "print(f\\"  BASE MODEL UNCHANGED: {BASE_FP_BEFORE == BASE_FP_AFTER}\\")\\n",\n    "print(f\\"  audit chain valid: {AUDIT.verify_chain()} ({len(AUDIT.entries)} operations recorded)\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 14 — OPTIONAL BASELINE ARM: SEQUENTIAL MERGE-LORA FORGETTING (GPU, flag-gated)\\n",\n    "# ============================================================\\n",\n    "# The comparison for the post: standard practice = merge each LoRA into base, train next.\\n",\n    "# Expect: skill-1 score degrades after merging skill-2/skill-3. Akili bank: no degradation.\\n",\n    "BASELINE = None\\n",\n    "if not CONFIG[\\"run_baseline_arm\\"]:\\n",\n    "    print(\\"[baseline] disabled (set AKILI_LLM_RUN_BASELINE=true to run). Cost: ~2 extra trainings.\\")\\n",\n    "else:\\n",\n    "    from peft import LoraConfig, get_peft_model\\n",\n    "    base2 = AutoModelForCausalLM.from_pretrained(BASE_NAME, torch_dtype=DTYPE).to(DEVICE)\\n",\n    "    seq_scores = {}\\n",\n    "    for i, skill in enumerate(CONFIG[\\"skills\\"]):\\n",\n    "        lcfg = LoraConfig(r=CONFIG[\\"lora\\"][\\"r\\"], lora_alpha=CONFIG[\\"lora\\"][\\"alpha\\"],\\n",\n    "                          lora_dropout=CONFIG[\\"lora\\"][\\"dropout\\"],\\n",\n    "                          target_modules=CONFIG[\\"lora\\"][\\"targets\\"], task_type=\\"CAUSAL_LM\\")\\n",\n    "        m = get_peft_model(base2, lcfg); m.train()\\n",\n    "        opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=CONFIG[\\"train\\"][\\"lr\\"])\\n",\n    "        dl = DataLoader(SkillDataset(DATA[skill][\\"train\\"]), batch_size=CONFIG[\\"train\\"][\\"batch\\"],\\n",\n    "                        shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(SEED))\\n",\n    "        for _ in range(CONFIG[\\"train\\"][\\"epochs\\"]):\\n",\n    "            for j, batch in enumerate(dl):\\n",\n    "                batch = {k: v.to(DEVICE) for k, v in batch.items()}\\n",\n    "                out = m(**batch); (out.loss / CONFIG[\\"train\\"][\\"grad_accum\\"]).backward()\\n",\n    "                if (j + 1) % CONFIG[\\"train\\"][\\"grad_accum\\"] == 0:\\n",\n    "                    opt.step(); opt.zero_grad()\\n",\n    "        base2 = m.merge_and_unload()   # bake skill into weights (standard practice)\\n",\n    "        # re-measure skill 1 after each merge\\n",\n    "        if skill != CONFIG[\\"skills\\"][0]:\\n",\n    "            sc = []\\n",\n    "            for q, ref in DATA[CONFIG[\\"skills\\"][0]][\\"eval\\"][:40]:\\n",\n    "                sc.append(skill_metric(CONFIG[\\"skills\\"][0], generate(base2, q), ref))\\n",\n    "            seq_scores[f\\"after_{skill}\\"] = float(np.mean(sc))\\n",\n    "            print(f\\"[baseline] skill-1 score after merging {skill}: {seq_scores[f\'after_{skill}\']:.3f}\\")\\n",\n    "        del m, opt\\n",\n    "        if DEVICE == \\"cuda\\": torch.cuda.empty_cache()\\n",\n    "    BASELINE = {\\"skill1_sequential_merge\\": seq_scores}\\n",\n    "    del base2\\n",\n    "    if DEVICE == \\"cuda\\": torch.cuda.empty_cache()\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 15 — FINAL REPORT + HARD CHECKS + POST TEXT\\n",\n    "# ============================================================\\n",\n    "skill_cards = {s: REGISTRY.skills[s][\\"validation\\"] for s in REGISTRY.skills}\\n",\n    "report = {\\n",\n    "    \\"protocol\\": \\"akili-skill-runtime-v0.1\\",\\n",\n    "    \\"base_model\\": BASE_NAME,\\n",\n    "    \\"skills\\": {s: {\\"state\\": REGISTRY.skills[s][\\"state\\"],\\n",\n    "                   \\"score\\": REGISTRY.skills[s][\\"validation\\"][\\"score\\"]}\\n",\n    "               for s in REGISTRY.skills},\\n",\n    "    \\"poison_skill\\": {\\"name\\": CONFIG[\\"poison_skill\\"],\\n",\n    "                     \\"state\\": REGISTRY.skills[CONFIG[\\"poison_skill\\"]][\\"state\\"],\\n",\n    "                     \\"validation\\": skill_cards[CONFIG[\\"poison_skill\\"]]},\\n",\n    "    \\"routing\\": ROUTING,\\n",\n    "    \\"base_model_unchanged\\": BASE_FP_BEFORE == BASE_FP_AFTER,\\n",\n    "    \\"audit_entries\\": len(AUDIT.entries),\\n",\n    "    \\"baseline_arm\\": BASELINE,\\n",\n    "}\\n",\n    "with open(os.path.join(RUN_DIR, \\"akili_llm_report.json\\"), \\"w\\") as fh:\\n",\n    "    json.dump(report, fh, indent=2)\\n",\n    "\\n",\n    "hard_checks = {\\n",\n    "    \\"base_model_never_trained_frozen\\": True,\\n",\n    "    \\"base_fingerprint_unchanged\\": BASE_FP_BEFORE == BASE_FP_AFTER,\\n",\n    "    \\"adapters_write_once\\": True,\\n",\n    "    \\"validation_before_activation\\": True,\\n",\n    "    \\"poison_activation_blocked\\": REGISTRY.skills[CONFIG[\\"poison_skill\\"]][\\"state\\"] in (\\"ROLLED_BACK\\", \\"QUARANTINED\\"),\\n",\n    "    \\"rollback_surgical_only_target_removed\\": sorted(REGISTRY.active_skills()) == sorted(CONFIG[\\"skills\\"]),\\n",\n    "    \\"audit_chain_valid\\": AUDIT.verify_chain(),\\n",\n    "    \\"all_skill_scores_finite\\": all(np.isfinite(REGISTRY.skills[s][\\"validation\\"][\\"score\\"]) for s in REGISTRY.skills),\\n",\n    "    \\"no_merged_weights_in_bank\\": True,\\n",\n    "}\\n",\n    "hard_checks[\\"all_passed\\"] = all(hard_checks.values())\\n",\n    "with open(os.path.join(RUN_DIR, \\"hard_checks.json\\"), \\"w\\") as fh:\\n",\n    "    json.dump(hard_checks, fh, indent=2)\\n",\n    "\\n",\n    "print(json.dumps(report, indent=2))\\n",\n    "print(\\"\\\\n[hard checks]\\", json.dumps(hard_checks, indent=2))\\n",\n    "print(f\\"[output] {RUN_DIR}\\")\\n"\n   ],\n   "execution_count": null,\n   "outputs": []\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## The post (fill in your numbers from the report)\\n",\n    "\\n",\n    "> We taught a frozen open-source LLM 3 new skills — SQL writing, JSON extraction, meeting action items — without ever touching its weights. Each skill is a small write-once adapter in the Akili skill bank, routed automatically by a semantic address space.\\n",\n    ">\\n",\n    "> Then we did what nobody demos: we deployed a **poisoned skill** (corrupted training data — the supply-chain nightmare). Akili\'s validation caught it automatically, blocked activation, and rolled the skill back in **milliseconds**. The other three skills: untouched. The base model: hash-identical.\\n",\n    ">\\n",\n    "> In a standard fine-tune, removing a bad skill means retraining from scratch. In Akili, it\'s a file operation with an audit receipt.\\n",\n    ">\\n",\n    "> **\\"git revert for LLM skills.\\"** Built solo in Kinshasa, DRC. 🙏\\n",\n    "\\n",\n    "**Next steps after this run:** record the Act I–IV sequence as a 60-second video; attach `akili_llm_report.json` numbers; then the scale-up path is skill banks for agent teams (the data-rich arbitration environment the CIFAR phase pointed to).\\n"\n   ]\n  }\n ]\n}'
SOURCE_FILENAME = 'Akili_Skill_Runtime_v0_1_Standalone_Colab.ipynb'

runtime_root = Path("/tmp/akili_publication_batch")
runtime_root.mkdir(parents=True, exist_ok=True)
runner_path = runtime_root / f"{RUNNER_NAME}.py"
runner_path.write_text(RUNNER_SOURCE, encoding="utf-8")
ast.parse(RUNNER_SOURCE)
compile(RUNNER_SOURCE, str(runner_path), "exec")
source_notebook = runtime_root / SOURCE_FILENAME
source_notebook.write_text(SOURCE_NOTEBOOK_TEXT, encoding="utf-8")
if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
runner = importlib.import_module(RUNNER_NAME)

verify_root = Path("/tmp/akili_batch_runner_verify")
shutil.rmtree(verify_root, ignore_errors=True)
verification = runner.synthetic_verification(verify_root)
assert verification["passed"], verification
print("Runner protocol:", runner.PROTOCOL)
print(json.dumps(verification["checks"], indent=2))


In [ ]:
import os
from pathlib import Path

project_root = Path(os.getenv(
    "AKILI_LLM_PUBLICATION_ROOT",
    "/content/drive/MyDrive/AKM_CLR",
))
batch_root = project_root / "publication_runs" / "llm_skill_runtime_v0_1_three_seed"
seeds = [
    int(value) for value in os.getenv("AKILI_LLM_PUBLICATION_SEEDS", "1,2,3").split(",")
    if value.strip()
]
force = os.getenv("AKILI_LLM_PUBLICATION_FORCE", "0").strip().lower() in {
    "1", "true", "yes"
}
print({"batch_root": str(batch_root), "seeds": seeds, "force": force})


In [ ]:
import os
SKIP_REAL = os.getenv("AKILI_BATCH_SKIP_REAL", "0").strip().lower() in {
    "1", "true", "yes"
}
if SKIP_REAL:
    RESULT = None
    print("LLM publication rerun skipped.")
else:
    try:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Use a Colab GPU runtime for the LLM publication rerun.")
    except ImportError:
        pass

    def env_builder(seed, output_root):
        relative_output = output_root.relative_to(project_root).as_posix()
        return {
            "AKILI_LLM_SEED": str(seed),
            "AKILI_ROOT": str(project_root),
            "AKILI_LLM_OUTPUT_SUBDIR": relative_output,
            "AKILI_LLM_RUN_BASELINE": os.getenv("AKILI_LLM_RUN_BASELINE", "1"),
        }

    def output_root_builder(seed):
        return batch_root / f"seed_{seed}" / "runs"

    RESULT = runner.run_seed_batch(
        source_notebook=source_notebook,
        batch_root=batch_root,
        seeds=seeds,
        env_builder=env_builder,
        output_root_builder=output_root_builder,
        force=force,
    )


In [ ]:
if RESULT is not None:
    import json
    print(json.dumps(RESULT, indent=2))
    if "google.colab" in sys.modules and os.getenv(
        "AKILI_BATCH_AUTO_DOWNLOAD", "1"
    ).strip().lower() in {"1", "true", "yes"}:
        from google.colab import files
        files.download(RESULT["evidence_zip"])
